# 01 — Exploratory Data Analysis
**Customer Churn Prediction — Banking**

Goal: Understand the dataset, identify churn patterns, and uncover key behavioral segments at risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
COLORS = {'churn': '#E63946', 'retain': '#457B9D', 'neutral': '#2B2D42'}

df = pd.read_csv('../data/Churn_Modelling.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Basic Overview

In [ ]:
# Drop irrelevant columns
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

print('=== Missing values ===')
print(df.isnull().sum())
print()
print('=== Data types ===')
print(df.dtypes)
print()
print('=== Summary statistics ===')
df.describe().round(2)

## 2. Churn Rate Overview

In [ ]:
churn_rate = df['Exited'].mean() * 100
counts = df['Exited'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Retained', 'Churned'], counts.values,
            color=[COLORS['retain'], COLORS['churn']], edgecolor='white', linewidth=1.5)
axes[0].set_title('Customer Retention vs Churn', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Retained', 'Churned'],
            colors=[COLORS['retain'], COLORS['churn']],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title(f'Churn Rate: {churn_rate:.1f}%', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Overall churn rate: {churn_rate:.1f}% — class imbalance present, will need SMOTE in modeling.')

## 3. Churn by Geography & Gender

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate by Geography
geo_churn = df.groupby('Geography')['Exited'].mean().sort_values(ascending=False) * 100
bars = axes[0].bar(geo_churn.index, geo_churn.values,
                   color=[COLORS['churn'], COLORS['neutral'], COLORS['retain']])
axes[0].set_title('Churn Rate by Geography', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 40)
for bar, val in zip(bars, geo_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')

# Churn rate by Gender
gender_churn = df.groupby('Gender')['Exited'].mean().sort_values(ascending=False) * 100
bars2 = axes[1].bar(gender_churn.index, gender_churn.values,
                    color=[COLORS['churn'], COLORS['retain']])
axes[1].set_title('Churn Rate by Gender', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 35)
for bar, val in zip(bars2, gender_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/churn_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Germany has significantly higher churn rate. Female customers churn more than male.')

## 4. Churn by Number of Products & Activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn by NumOfProducts
prod_churn = df.groupby('NumOfProducts')['Exited'].mean() * 100
bars = axes[0].bar(prod_churn.index.astype(str), prod_churn.values,
                   color=[COLORS['retain'], COLORS['neutral'], COLORS['churn'], COLORS['churn']])
axes[0].set_title('Churn Rate by Number of Products', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Products')
axes[0].set_ylabel('Churn Rate (%)')
for bar, val in zip(bars, prod_churn.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', fontweight='bold')

# Churn by IsActiveMember
active_churn = df.groupby('IsActiveMember')['Exited'].mean() * 100
bars2 = axes[1].bar(['Inactive', 'Active'], active_churn.values,
                    color=[COLORS['churn'], COLORS['retain']])
axes[1].set_title('Churn Rate by Account Activity', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 35)
for bar, val in zip(bars2, active_churn.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/churn_products_activity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Customers with 3-4 products churn at extremely high rates. Inactive members churn ~2x more.')

## 5. Age Distribution by Churn Status

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

df[df['Exited']==0]['Age'].plot(kind='hist', bins=40, alpha=0.6,
                                 color=COLORS['retain'], label='Retained', ax=ax)
df[df['Exited']==1]['Age'].plot(kind='hist', bins=40, alpha=0.6,
                                 color=COLORS['churn'], label='Churned', ax=ax)

ax.set_title('Age Distribution: Churned vs Retained Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.legend(fontsize=12)

# Annotate median ages
med_retain = df[df['Exited']==0]['Age'].median()
med_churn = df[df['Exited']==1]['Age'].median()
ax.axvline(med_retain, color=COLORS['retain'], linestyle='--', linewidth=2, label=f'Median retained: {med_retain}')
ax.axvline(med_churn, color=COLORS['churn'], linestyle='--', linewidth=2, label=f'Median churned: {med_churn}')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Median age — Retained: {med_retain}, Churned: {med_churn}')
print('Key insight: Churned customers are significantly older on average. Peak churn in 40-60 age group.')

## 6. Correlation Heatmap

In [ ]:
# Encode categoricals for correlation
df_encoded = df.copy()
df_encoded['Geography'] = df_encoded['Geography'].map({'France': 0, 'Germany': 1, 'Spain': 2})
df_encoded['Gender'] = df_encoded['Gender'].map({'Male': 0, 'Female': 1})

corr = df_encoded.corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — All Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: Age shows strongest positive correlation with churn. IsActiveMember shows negative correlation.')

## EDA Summary

| Finding | Business Implication |
|---|---|
| Churn rate = 20.4% — class imbalance | Need SMOTE before modeling |
| Germany churn rate ~32% vs France ~16% | Region-specific retention strategy needed |
| Female customers churn more (25% vs 16%) | Gender-targeted retention offers |
| Customers with 3-4 products: churn >80% | Over-selling products damages retention |
| Inactive customers churn 2x more | Reactivation campaigns are critical |
| Churned customers are ~10 years older | Age-based segmentation for outreach |

→ **Next step:** Feature engineering + preprocessing in `02_preprocessing.ipynb`